## Setup for Forest Tree

In [1]:
# --- Setup: load our toolbox ---
import numpy as np                     # numpy = calculator for lists of numbers
import pandas as pd                    # pandas = tool for working with tables of data (like Excel)
import matplotlib.pyplot as plt        # matplotlib = drawing charts
import seaborn as sns                  # seaborn = prettier charts, built on matplotlib
import warnings                        # lets us hide distracting technical warnings

# Tools from scikit-learn (the machine-learning library):
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold  # split data + fair testing
from sklearn.compose import ColumnTransformer          # apply different prep steps to different columns
from sklearn.pipeline import Pipeline                  # chain prep + model into one tidy assembly line
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # put numbers on same scale; turn categories into numbers
from sklearn.impute import SimpleImputer               # fill in missing values sensibly
from sklearn.linear_model import LogisticRegression    # previous baseline model
from sklearn.tree import DecisionTreeClassifier        # today's new model: the decision tree
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score  # scorecards for models

warnings.filterwarnings("ignore")          # hide harmless warning messages so output stays readable
sns.set_style("whitegrid")                 # charts get a light grid background
plt.rcParams["figure.figsize"] = (11, 4.5) # default chart size (width, height in inches)

print("✅ Libraries loaded — ready to compare baseline vs trees")

✅ Libraries loaded — ready to compare baseline vs trees


## Step 1 — Reload the dataset and the L03 pipeline

In [2]:
# --- Load the fraud data and split it, exactly as we did in L02 ---
df = pd.read_csv("data/credit_card_fraud_2026.csv")   # read the customer table from file
y  = df["is_fraud"]                              # y = the answer we want to predict (did they commit fraud? 1/0)
X  = df.drop(columns=["transaction_id", "is_fraud"])# X = the clues (everything except the ID and the answer)

# Split into training data (model learns from this) and test data (kept hidden, used to grade the model)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,  # 20% held out; stratify keeps fraud % equal in both halves; random_state makes the split repeatable
)

# List which columns are numbers vs categories — they need different preparation
numeric_features = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
categorical_features = [
    "merchant_category", 
    "card_type", 
    "auth_method", 
    "channel", 
    "device_type", 
    "is_foreign_transaction", 
    "is_new_merchant", 
    "used_vpn", 
    "ip_country_mismatch", 
    "billing_shipping_mismatch", 
    "is_ai_generated_scam_attempt",
]

# Preprocessor = the data-prep recipe (same one from NB03):
preprocessor = ColumnTransformer([
    # For number columns: fill gaps with the median, then rescale so no column dominates
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    # For category columns: fill gaps with the most common value, then turn each category into 0/1 columns
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])
print(f"Training set: {len(X_train):,} customers ({y_train.mean():.1%} fraud)")
print(f"Test set:     {len(X_test):,} customers ({y_test.mean():.1%} fraud)")

Training set: 16,000 customers (1.7% fraud)
Test set:     4,000 customers (1.7% fraud)


## Step 2 — Re-train the logistic regression baseline

In [3]:
# --- Rebuild the NB02 baseline: this is the score every new model must beat ---
logreg_pipe = Pipeline([
    ("prep",  preprocessor),                                      # step 1: prepare the data
    ("model", LogisticRegression(max_iter=1000, random_state=42)),# step 2: the model
])
logreg_pipe.fit(X_train, y_train)   # learn from the training customers

# Predict probabilities, apply the threshold Sarah recommended in NB03
y_proba_lr = logreg_pipe.predict_proba(X_test)[:, 1]  # for each test customer: probability of fraud (0 to 1)
y_pred_lr  = (y_proba_lr >= 0.25).astype(int)         # flag anyone above 25% risk (lower bar = catch more fraudsters)

# Print the scorecard for the baseline
print("=== baseline (LogisticRegression, threshold = 0.25) ===")
print(f"Accuracy:   {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Precision:  {precision_score(y_test, y_pred_lr):.3f}")
print(f"Recall:     {recall_score(y_test, y_pred_lr):.3f}")
print(f"F1:         {f1_score(y_test, y_pred_lr):.3f}")
print(f"Flagged:    {int(y_pred_lr.sum())} customers")
print()
print(f"→ This is THE BAR. Every model today has to beat F1 ≈ {f1_score(y_test, y_pred_lr):.3f}.")

=== baseline (LogisticRegression, threshold = 0.25) ===
Accuracy:   0.985
Precision:  0.569
Recall:     0.485
F1:         0.524
Flagged:    58 customers

→ This is THE BAR. Every model today has to beat F1 ≈ 0.524.


## Step 3 — One naive decision tree

In [4]:
# --- First attempt: a single decision tree, left to grow as deep as it wants ---
tree_pipe = Pipeline([
    ("prep",  preprocessor),                                   # same data prep as before
    ("model", DecisionTreeClassifier(random_state=42)),        # default: grow until pure (memorises the training data)
])
tree_pipe.fit(X_train, y_train)   # let the tree learn — it will keep splitting until every training customer is "explained"

# Tree's predict() uses threshold=0.5 internally on probabilities derived from leaf purity
y_pred_tree_train = tree_pipe.predict(X_train)  # score it on customers it has already SEEN
y_pred_tree_test  = tree_pipe.predict(X_test)   # score it on customers it has NEVER seen — the honest test

# Compare train vs test accuracy — a big gap means the tree memorised instead of learned
print("=== Single decision tree (full depth) ===")
print(f"TRAIN accuracy:  {accuracy_score(y_train, y_pred_tree_train):.3f}")
print(f"TEST  accuracy:  {accuracy_score(y_test,  y_pred_tree_test):.3f}")
print(f"Gap:             {accuracy_score(y_train, y_pred_tree_train) - accuracy_score(y_test, y_pred_tree_test):.3f}")
print()
print(f"TEST F1:    {f1_score(y_test, y_pred_tree_test):.3f}")
print(f"TEST recall:{recall_score(y_test, y_pred_tree_test):.3f}")
print(f"TEST prec:  {precision_score(y_test, y_pred_tree_test):.3f}")

=== Single decision tree (full depth) ===
TRAIN accuracy:  1.000
TEST  accuracy:  0.971
Gap:             0.029

TEST F1:    0.204
TEST recall:0.221
TEST prec:  0.190


## Step 4 — A first shallow tree (just to show the bias-variance tradeoff)

In [5]:
# --- Second attempt: a SHALLOW tree — only 3 yes/no questions deep ---
shallow_tree = Pipeline([
    ("prep",  preprocessor),                                            # same data prep
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),    # max_depth=3 stops the tree from memorising
])
shallow_tree.fit(X_train, y_train)   # learn again, but with the depth limit on

y_pred_st_train = shallow_tree.predict(X_train)  # score on seen customers
y_pred_st_test  = shallow_tree.predict(X_test)   # score on unseen customers

# This time train and test should be close — but is the score good enough?
print("=== Shallow tree (max_depth=3) ===")
print(f"TRAIN accuracy:  {accuracy_score(y_train, y_pred_st_train):.3f}")
print(f"TEST  accuracy:  {accuracy_score(y_test,  y_pred_st_test):.3f}")
print(f"Gap:             {accuracy_score(y_train, y_pred_st_train) - accuracy_score(y_test, y_pred_st_test):.3f}")
print()
print(f"TEST F1:    {f1_score(y_test, y_pred_st_test):.3f}")
print()
print("→ Train and test agree (no overfitting).")
print("→ But the F1 is lower than the deep tree's TEST F1 — the shallow tree can't capture the patterns.")
print()
print("This is the BIAS-VARIANCE tradeoff in one notebook:")
print("  Deep tree:    low bias, HIGH variance (overfits)")
print("  Shallow tree: HIGH bias, low variance (underfits)")
print("  Random Forest will give us low bias AND low variance — by averaging many deep trees.")

=== Shallow tree (max_depth=3) ===
TRAIN accuracy:  0.983
TEST  accuracy:  0.983
Gap:             0.000

TEST F1:    0.000

→ Train and test agree (no overfitting).
→ But the F1 is lower than the deep tree's TEST F1 — the shallow tree can't capture the patterns.

This is the BIAS-VARIANCE tradeoff in one notebook:
  Deep tree:    low bias, HIGH variance (overfits)
  Shallow tree: HIGH bias, low variance (underfits)
  Random Forest will give us low bias AND low variance — by averaging many deep trees.


## ✅ Section Summary

| Model | Train acc | Test acc | Test F1 | Diagnosis |
|---|---|---|---|---|
| **L03 Logistic Regression @ 0.25** | — | ~0.98 | ~0.52 | The bar to beat |
| **Single decision tree (full depth)** | ~1.00 | ~0.97 | varies | OVERFITS — memorises training data |
| **Shallow tree (max_depth=3)** | ~0.98 | ~0.98 | very low | UNDERFITS — too simple |

**Key insight:**
> One tree alone is never the answer. Either it overfits (full depth) or it underfits (shallow). Random Forest fixes this by AVERAGING many deep trees, each trained on slightly different data. The variance washes out; the bias stays low.